In [10]:
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
# Download latest version
path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset",output_dir='../data/external',force_download=True)

print("Path to dataset files:", path)

100%|██████████████████████████████████████| 0.98M/0.98M [00:00<00:00, 1.52MB/s]

Extracting files...
Path to dataset files: ../data/external


In [11]:
import pandas as pd
df=pd.read_csv(path+'/UCI_Credit_Card.csv')

In [12]:
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default.payment.next.month'],
      dtype='str')

## Обучим модель v1 самую простую

In [13]:
y = df['default.payment.next.month']

# X - признаки (фичи).
# Столбец 'ID' является идентификатором и не несет прогностической ценности,
# поэтому его необходимо исключить из обучающих данных.
X = df.drop(columns=['default.payment.next.month', 'ID'])

# 1. Разделение данных на обучающую и тестовую выборки
# stratify=y гарантирует, что соотношение классов в выборках будет таким же, как в исходных данных.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
from sklearn.linear_model import LogisticRegression


clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# 3. Предсказание и оценка модели
# Метод predict() возвращает предсказанные классы (0 или 1)
y_pred = clf.predict(X_test)

## LinearRegression(copy_X=True, fit_intercept=True, n_jobs=None, normalize=False)

/home/daniil/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(6000,))

In [16]:
y_test

6907     0
24575    0
26766    0
2156     1
3179     0
        ..
8836     0
1259     0
27309    0
29583    0
24399    0
Name: default.payment.next.month, Length: 6000, dtype: int64

In [17]:
print("Оценка на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Оценка на тестовой выборке:
Accuracy: 0.8073


In [18]:
import pickle

with open('../models/model_1.pkl', 'wb') as output:
    pickle.dump(clf, output)

## Обучим модель v2 дерево

In [19]:
from sklearn.ensemble import RandomForestClassifier
import pickle # Импортируем pickle вместо joblib

# ... (предполагается, что X_train, y_train, X_test, y_test уже определены)

# 2. Создание и обучение модели
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 3. Оценка качества модели (этот блок остается без изменений)
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test)
print("Оценка на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 4. Сохранение обученной модели в файл формата .pkl
model_filename = '../models/model_2.pkl'

# Открываем файл в режиме записи в двоичном формате ('wb')
with open(model_filename, 'wb') as file:
    pickle.dump(clf, file)

print(f"\nМодель успешно сохранена в файл: {model_filename}")

Оценка на тестовой выборке:
Accuracy: 0.8120

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.94      0.89      4673
           1       0.63      0.36      0.46      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.65      0.67      6000
weighted avg       0.79      0.81      0.79      6000


Модель успешно сохранена в файл: ../models/model_2.pkl


In [20]:
from flask import Flask, request, jsonify
import pickle
import numpy as np
import os
import glob
import logging